In [ ]:
import os
import re
import sys
import numpy as np
import pandas as pd
import xarray as xr
from natsort import natsorted
from scipy.stats import zscore
from os.path import join as pjoin
import plotly.graph_objects as go
from scipy.ndimage import gaussian_filter

sys.path.append('../')
import mannkendall as mk
import plotting_functions as pf
import circletrack_neural as ctn

In [ ]:
## Settings
project_dir = 'MultiCon_Imaging'
experiment_dir = 'MultiCon_Imaging5'
output_path = os.path.abspath(f'../../{project_dir}/{experiment_dir}/output')
fig_path = '../../../Manuscripts/MultiCon/intermediate_plots/qc_plots'
mouse = 'mc44'
alpha = 0.001 ## for Mann-Kendall trend test

### Create a plot of a dropped neruon in one column and a kept neuron in another.

In [ ]:
## Can use this code block to check individual sessions and QC by eye units that the Mann-Kendall trend test is detecting to be dropped
session_idx = 11
c_path = pjoin(output_path, f'aligned_minian/{mouse}/C')
yra_path = pjoin(output_path, f'aligned_minian/{mouse}/YrA')
s_path = pjoin(output_path, f'aligned_minian/{mouse}/S')
place_path = pjoin(output_path, f'place_cells/{mouse}/S')

c_save = pjoin(output_path, f'aligned_minian_two/{mouse}/C')
yra_save = pjoin(output_path, f'aligned_minian_two/{mouse}/YrA')
s_save = pjoin(output_path, f'aligned_minian_two/{mouse}/S')
place_save = pjoin(output_path, f'place_cells_two/{mouse}/S')
for idx, session in enumerate(natsorted(os.listdir(c_path))):
    if idx == session_idx:
        day = re.search('_([0-9]+)', session)[0]
        fig = pf.custom_graph_template(x_title='Time (s)', y_title='Activity (a.u.)', width=700)
        cell_trends = {'unit_id': [], 'trend': [], 'mk_score': [], 'pval': []}

        ## Load data
        C = xr.open_dataset(pjoin(c_path, session))['C']
        if len(day) == 2:
            yra_string = f'{mouse}_YrA_{day[-1]}.nc'
            s_string = f'{mouse}_S_{day[-1]}.nc'
            c_string = f'{mouse}_C_{day[-1]}.nc'
        else:
            yra_string = f'{mouse}_YrA_{day[-2:]}.nc'
            s_string = f'{mouse}_S_{day[-2:]}.nc'
            c_string = f'{mouse}_C_{day[-2:]}.nc'
        YrA = xr.open_dataset(pjoin(yra_path, yra_string))['YrA']
        S = xr.open_dataset(pjoin(s_path, s_string))['S']
        S_place = xr.open_dataset(pjoin(place_path, s_string))['S']

        ## Mann-Kendall trend test on taking the minimum value in a 20-second bin of z-scored data
        ## Rationale is that if there is an increasing/decreasing trend, the z-scored values get brought above or
        ## below zero, and Mann-Kendall can detect this. 20-second bin allows for less false positives
        cell_act = C.values
        zdata = zscore(cell_act, axis=1)
        bin_act = ctn.bin_activity(zdata, bin_size_seconds=20, func=np.min)
        for uid in np.arange(0, C.shape[0]):
            mkt = mk.original_test(bin_act[uid, :])
            cell_trends['unit_id'].append(uid)
            cell_trends['trend'].append(mkt.trend)
            cell_trends['mk_score'].append(np.abs(mkt.s))
            cell_trends['pval'].append(mkt.p)
        trends = pd.DataFrame(cell_trends)
        dropped_uids = trends[(trends['pval'] < alpha)].reset_index(drop=True)

        for neuron in dropped_uids['unit_id']:
            fig.add_trace(go.Scattergl(x=YrA['behav_t'].values, y=YrA.values[neuron, :], mode='lines', visible='legendonly',
                                    line_color='black', name=f'Neuron {neuron}', showlegend=False, legendgroup=f'{neuron}'))
            fig.add_trace(go.Scattergl(x=C['behav_t'].values, y=zdata[neuron, :], mode='lines', visible='legendonly',
                                    line_color='red', name=f'Neuron {neuron}', legendgroup=f'{neuron}'))
    else:
        pass
fig.show()

In [ ]:
## Save an example neuron that should be dropped for Supplemental Fig
dropped_neuron = 637 ## for mc51
kept_neuron = 380 ## for mc51
fig = pf.custom_graph_template(x_title='Time (s)', y_title='', width=800, titles=['Dropped Neuron', 'Kept Neuron'],
                               rows=1, columns=2, shared_x=True, shared_y=True)
fig.add_trace(go.Scattergl(x=YrA['behav_t'].values, y=YrA.values[dropped_neuron, :] / np.max(YrA.values[dropped_neuron, :]), mode='lines',
                           line_color='black', name='Raw F', showlegend=False), row=1, col=1)
fig.add_trace(go.Scattergl(x=YrA['behav_t'].values, y=YrA.values[kept_neuron, :] / np.max(YrA.values[kept_neuron, :]), mode='lines',
                           line_color='black', name='Raw F', showlegend=False), row=1, col=2)
fig.update_yaxes(title='Normalized Activity (a.u.)', col=1)
fig.show()
fig.write_image(pjoin(fig_path, 'dropped_kept_neurons.svg'), width=800, height=500)

### Create a plot of the S-matrix on top of YrA showing before an after Otsu thresholding.

In [ ]:
## Explore
fig = pf.custom_graph_template(x_title='Time (s)', y_title='', rows=1, columns=2, shared_x=True, shared_y=True,
                               titles=['Before Thresholding', 'After Thresholding'], width=800)

neuron = 73
single_neuron_thresh = ctn.calculate_otsu_thresh(S[neuron, :])
scale_factor = 10

for col_val in [1, 2]:
    fig.add_trace(go.Scattergl(x=YrA['behav_t'].values, y=YrA.values[neuron, :], mode='lines', line_color='black', 
                               name=f'Neuron {neuron}', showlegend=False, legendgroup=f'{neuron}'), row=1, col=col_val)

    if col_val == 1:
        output = S.copy()
    elif col_val == 2:
        output = S.copy()
        output[neuron, output[neuron, :] <= (single_neuron_thresh / 2)] = 0

    fig.add_trace(go.Scattergl(x=S['behav_t'].values, y=output[neuron, :] * scale_factor, mode='lines', line_color='red', 
                            name=f'Neuron {neuron}', legendgroup=f'{neuron}', showlegend=False, opacity=0.8), row=1, col=col_val)
fig.show()

In [ ]:
## for mc44, use neuron 3 from session 11 and plot up to 150s
fig = pf.custom_graph_template(x_title='Time (s)', y_title='', rows=1, columns=2, shared_x=True, shared_y=True,
                               titles=['Before Thresholding', 'After Thresholding'], width=800)

neuron = 3
single_neuron_thresh = ctn.calculate_otsu_thresh(S[neuron, :])
scale_factor = 10
sub_x = YrA['behav_t'].values < 150

for col_val in [1, 2]:
    fig.add_trace(go.Scattergl(x=YrA['behav_t'].values[sub_x], y=YrA.values[neuron, :][sub_x], mode='lines', line_color='black', 
                               name=f'Neuron {neuron}', showlegend=False, legendgroup=f'{neuron}'), row=1, col=col_val)

    if col_val == 1:
        output = S.copy()
    elif col_val == 2:
        output = S.copy()
        output[neuron, output[neuron, :] <= (single_neuron_thresh / 2)] = 0

    fig.add_trace(go.Scattergl(x=S['behav_t'].values[sub_x], y=output[neuron, :][sub_x] * scale_factor, mode='lines', line_color='red', 
                            name=f'Neuron {neuron}', legendgroup=f'{neuron}', showlegend=False, opacity=0.8), row=1, col=col_val)
fig.update_yaxes(title='Activity (a.u.)')
fig.show()
fig.write_image(pjoin(fig_path, f'{mouse}_neuron{neuron}_before_after_otsu.svg'), width=800, height=500)

### Code below is for checking how trials are being labeled on the behavior.

In [ ]:
mouse = 'mc51' 
project = 'MultiCon_Imaging' 
experiment = 'MultiCon_Imaging5'
day = 16 
session = f'{mouse}_{day}.feat'
jump_val = 295 
angle_accumulation = -294
min_trial_length = 171 
max_trial_length = 750
convert_to_rad = False
behav = pd.read_feather(f'../../{project}/{experiment}/output/behav/{mouse}/{session}')
angular_position = behav['a_pos'].to_numpy()


trials = np.full(len(angular_position), np.nan)
if convert_to_rad:
    lin_pos = angular_position * (np.pi/180)
else:
    lin_pos = angular_position

prev_idx = 0
diffs = np.diff(lin_pos)
for trial, jump in enumerate(np.where((diffs >= jump_val) | (diffs <= -jump_val))[0]):
    ## Account for first trial
    if trial == 0:
        trials[prev_idx:jump + 1] = trial
        prev_idx = jump + 1
        prev_trial = trial ## create prev_trial variable to account for some "trials" in the loop not meeting trial criteria
    else:
        if prev_idx == jump: ## account for edge cases
            trials[prev_idx:jump + 1] = prev_trial
        elif (np.cumsum(diffs[prev_idx:jump])[-1] <= angle_accumulation) and (len(diffs[prev_idx:jump]) >= min_trial_length):
            trials[prev_idx:jump + 1] = prev_trial + 1
            prev_idx = jump + 1
            prev_trial = prev_trial + 1
        elif (np.cumsum(diffs[prev_idx:jump])[-1] >= abs(angle_accumulation)) and (len(diffs[prev_idx:jump]) >= min_trial_length):
            trials[prev_idx:jump + 1] = prev_trial + 1
            prev_idx = jump + 1
            prev_trial = prev_trial + 1
        elif diffs[prev_idx:jump].shape[0] >= max_trial_length:
            trials[prev_idx:jump + 1] = prev_trial + 1
            prev_idx = jump + 1
            prev_trial = prev_trial + 1
        else:
            ## If the data points don't meet the above criteria, set those data points as part of the previous trial
            trials[prev_idx:jump + 1] = prev_trial
            prev_idx = jump + 1
    
    ## Account for last trial
    if trial == len(np.where((diffs >= jump_val) | (diffs <= -jump_val))[0]) - 1:
        trials[prev_idx:] = prev_trial + 1

In [ ]:
from plotly.colors import sequential

cscale = sequential.Magma[1:-2]
fig = pf.custom_graph_template(x_title='Time (s)', y_title='Position (degrees)', width=1000, titles=[f'Day {day}'])
## Plot trials
for trial in behav['trials'].unique():
    tdata = behav[behav['trials'] == trial]
    lin_pos = gaussian_filter(tdata['a_pos'], sigma=2)
    
    fig.add_trace(go.Scatter(x=tdata['t'], y=lin_pos, mode='markers', marker_size=3, marker_color=cscale[int(trial % len(cscale))],
                            legendgroup='trials', name=f'Trial {trial}', showlegend=False))
    fig.add_trace(go.Scatter(x=tdata['t'][tdata['water']], y=lin_pos[tdata['water']], mode='markers', marker_color='red', marker_size=8,
                             showlegend=False, marker=dict(line=dict(width=1.5, color='black')), marker_symbol='diamond', name='Reward'))
    fig.add_trace(go.Scatter(x=tdata['t'][tdata['lick_port'] != -1], y=lin_pos[tdata['lick_port'] != -1], mode='markers', marker_color='darkgrey', marker_size=6,
                                 showlegend=False, marker=dict(line=dict(width=1.5, color='black')), marker_symbol='circle', name='Lick', opacity=0.7))
fig.data[100]['showlegend'] = True
fig.data[101]['showlegend'] = True
fig.show()
fig.write_image(pjoin(fig_path, f'{mouse}_day{day}_lin_pos_trials_rewards_licks.svg'), width=1000, height=500)